In [ ]:
# ======================================================
# Notebook: Drug Combination Optimisation (SVM)
# Maximise (- side effects)
# ======================================================

import numpy as np
from sklearn.svm import SVC

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,3)
y_raw = np.load("/mnt/data/initial_outputs.npy") # (N,)

# Transform objective (minimise side effects -> maximise negative)
y_transformed = -y_raw

# Convert to binary (best vs rest)
threshold = np.percentile(y_transformed, 75)
y_bin = (y_transformed >= threshold).astype(int)

# Train SVM surrogate
model = SVC(kernel="rbf", probability=True)
model.fit(X, y_bin)

# Candidate grid within bounds
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(3)]
grid = [np.linspace(b[0], b[1], 20) for b in bounds]
X_grid = np.array(np.meshgrid(*grid)).T.reshape(-1,3)

# Predict probability of best region
probs = model.predict_proba(X_grid)[:,1]

# Select next (10,3)
top_idx = np.argsort(probs)[-10:]
next_points = X_grid[top_idx]

print("Next (10,3) compound combinations:")
print(next_points)